In [1]:
import pandas as pd
from scipy import stats
import numpy as np
from pandas.api.types import CategoricalDtype
import re

In [2]:
PRTR_YEAR = 2025
POWER_YEAR = 2025

In [3]:
plants = pd.read_csv("plants_2.csv")
power = pd.read_csv("../production-v2/yearly.csv")

In [4]:
power2 = pd.read_csv("../production-v2/yearly.csv")

In [5]:
#power.melt(id_vars=['plantid'], value_vars=['yearpower', 'year'])

In [6]:
#power.loc[power.plantid == "06-05-100-0248923"]

In [7]:
#power.to_csv("powertest.csv")
#power2.to_csv("powertest2.csv")

In [8]:
#power2.sort_values(by=["year", "plantid"], ascending=False)

In [9]:
power.sort_values(by=["year", "plantid"], ascending=False)

,year,plantid,yearpower
665,2025,ST100125,3085432
671,2025,SN80011277,7583580
660,2025,SN70015796,12000836
662,2025,RP5000671,2119912
670,2025,NW900-9141660,2844960
...,...,...,...
43,2015,BB45025564,19541132
25,2015,BB23020490,1627136
13,2015,06-05-100-0853075,1014369
14,2015,06-05-100-0431554,3725774


In [10]:
powers = power.pivot(index='plantid', columns='year')

In [11]:
powers

yearpower                                                  \
year                     2015        2016        2017        2018        2019   
plantid                                                                         
06-02-B10117A007    1108161.0   1306733.0   1249523.0    936518.0    790997.0   
06-05-100-0431554   3725774.0   4348854.0   1140029.0         NaN         NaN   
06-05-100-0853075   1014369.0    765838.0    262118.0         NaN         NaN   
BB23020490          1627136.0   1504068.0   1549308.0   1436024.0   1329740.0   
BB45025564         19541132.0  19873060.0  19547884.0  18840828.0  15048036.0   
...                       ...         ...         ...         ...         ...   
SL0105352-G           33579.0    156589.0    218460.0    205599.0    285545.0   
SN70015796         17824456.0  16892980.0  17385608.0  17238224.0  16920124.0   
SN80011256           298386.0    574814.0    441036.0    337447.0    421516.0   
SN80011277         10794404.0  11433348.0  11882276.0  12032376.0   9265124.0   
ST100125            4149208.0   4249340.0   4293276.0   4881456.0   2887548.0   

                                                                               \
year                     2020        2021        2022        2023        2024   
plantid                                                                         
06-02-B10117A007     798187.0    842285.0    952563.0    707521.0    637327.0   
06-05-100-0431554         NaN         NaN         NaN         NaN         NaN   
06-05-100-0853075         NaN         NaN         NaN         NaN         NaN   
BB23020490          1365516.0   1246616.0   1332972.0   1163124.0   1376184.0   
BB45025564         11641296.0  12903344.0  12458764.0  11578228.0   9558780.0   
...                       ...         ...         ...         ...         ...   
SL0105352-G          235451.0    252929.0    172319.0    141928.0    137404.0   
SN70015796         13788512.0  14117384.0  17328260.0  11806900.0  12663172.0   
SN80011256           516161.0    220020.0         NaN         NaN         NaN   
SN80011277          8355324.0  11259204.0  12105444.0   7719272.0   5886812.0   
ST100125            2068592.0   3247588.0   3166456.0   3047344.0   3308936.0   

                               
year                     2025  
plantid                        
06-02-B10117A007          NaN  
06-05-100-0431554         NaN  
06-05-100-0853075         NaN  
BB23020490          1042356.0  
BB45025564          7932020.0  
...                       ...  
SL0105352-G               NaN  
SN70015796         12000836.0  
SN80011256                NaN  
SN80011277          7583580.0  
ST100125            3085432.0  

[73 rows x 11 columns]

In [12]:
df = powers.xs(('yearpower'), axis=1, drop_level=True)

In [13]:
df2 = df.reset_index()

In [14]:
df2

year,plantid,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,06-02-B10117A007,1108161.0,1306733.0,1249523.0,936518.0,790997.0,798187.0,842285.0,952563.0,707521.0,637327.0,NaN
1,06-05-100-0431554,3725774.0,4348854.0,1140029.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,06-05-100-0853075,1014369.0,765838.0,262118.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BB23020490,1627136.0,1504068.0,1549308.0,1436024.0,1329740.0,1365516.0,1246616.0,1332972.0,1163124.0,1376184.0,1042356.0
4,BB45025564,19541132.0,19873060.0,19547884.0,18840828.0,15048036.0,11641296.0,12903344.0,12458764.0,11578228.0,9558780.0,7932020.0
...,...,...,...,...,...,...,...,...,...,...,...,...
68,SL0105352-G,33579.0,156589.0,218460.0,205599.0,285545.0,235451.0,252929.0,172319.0,141928.0,137404.0,NaN
69,SN70015796,17824456.0,16892980.0,17385608.0,17238224.0,16920124.0,13788512.0,14117384.0,17328260.0,11806900.0,12663172.0,12000836.0
70,SN80011256,298386.0,574814.0,441036.0,337447.0,421516.0,516161.0,220020.0,NaN,NaN,NaN,NaN
71,SN80011277,10794404.0,11433348.0,11882276.0,12032376.0,9265124.0,8355324.0,11259204.0,12105444.0,7719272.0,5886812.0,7583580.0


In [15]:
#df

In [16]:
#powers.unstack('plantid')

In [17]:
np = pd.merge(plants, df2, how='left', on="plantid")

In [18]:
np

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,NW100-0046639,0065 - STW - BHKW - KWK,Baden-Württemberg,Erdgas,Ja,2024,2024,7.257,in Betrieb,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SD662-99,bnBm Gaskraftwerk Leipheim,Bayern,Erdgas,Nein,2023,2023,310.000,bnBm,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SD662-98,bnBm GasKW BIB GT8,Hessen,Erdgas,Nein,2023,2023,378.400,bnBm,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NW500-0875785,"Kraftwerk VII, Block 71, GT711",Baden-Württemberg,Erdgas,Ja,2022,2022,273.994,in Betrieb,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TH72012874,Gasmotor 5 Jena,Baden-Württemberg,Erdgas,Ja,2022,2022,60.875,in Betrieb,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,NW300-9046797,LEV KWG G19,Baden-Württemberg,Steinkohle,Ja,1999,1955,128.000,in Betrieb,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
308,NW300-0877384,Weisweiler,Baden-Württemberg,Braunkohle,Ja,2006,1955,2619.000,in Betrieb,8,...,13326683.0,15225905.0,13214489.0,10691896.0,9741967.0,11518803.0,12110914.0,7312086.0,8192538.0,NaN
309,BYS00334,Turbine 7,Bayern,Erdgas,Ja,2014,1952,54.795,in Betrieb,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
310,SD664-07,Fab Fortuna,Baden-Württemberg,Braunkohle,Ja,1948,1948,15.000,in Betrieb,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
np.sort_values(by=[2019], ascending=False)

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
249,NW100-0248923,Neurath F,Baden-Württemberg,Braunkohle,Ja,2012,1972,4211.000,in Betrieb,7,...,25468598.0,27112376.0,29463725.0,20841356.0,17221819.0,20015332.0,22326897.0,15021992.0,12629202.0,NaN
214,SN70015796,Boxberg Block Q,Sachsen,Braunkohle,Ja,2012,1979,2470.000,in Betrieb,4,...,16892980.0,17385608.0,17238224.0,16920124.0,13788512.0,14117384.0,17328260.0,11806900.0,12663172.0,12000836.0
210,BB45025564,Kraftwerk Jänschwalde Block F,Brandenburg,Braunkohle,Ja,1989,1981,3000.000,in Betrieb,6,...,19873060.0,19547884.0,18840828.0,15048036.0,11641296.0,12903344.0,12458764.0,11578228.0,9558780.0,7932020.0
289,NW300-0326774,Niederaußem,Baden-Württemberg,Braunkohle,Ja,2003,1963,3359.000,in Betrieb,8,...,14989316.0,19417424.0,18215928.0,12799092.0,8126548.0,12598900.0,15110432.0,11693500.0,10406156.0,10038940.0
169,SD666-16,Isar 2,Bayern,Kernenergie,Nein,1988,1988,1410.000,stillgelegt,1,...,11334924.0,10900504.0,11475721.0,11382289.0,11028613.0,11349943.0,11610304.0,2831770.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306,BYS00925,Sammelschienen-Kraftwerk Gas- und Dampfturbinen,Bayern,Erdgas,Ja,1956,1956,112.719,in Betrieb,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
307,NW300-9046797,LEV KWG G19,Baden-Württemberg,Steinkohle,Ja,1999,1955,128.000,in Betrieb,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
309,BYS00334,Turbine 7,Bayern,Erdgas,Ja,2014,1952,54.795,in Betrieb,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
310,SD664-07,Fab Fortuna,Baden-Württemberg,Braunkohle,Ja,1948,1948,15.000,in Betrieb,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
#np

In [21]:
co2dict2 = {}
for j in range(2007, PRTR_YEAR):
    co2dict2[j] = 'co2_' + str(j)

In [22]:
co2dict2

{2007: 'co2_2007',
 2008: 'co2_2008',
 2009: 'co2_2009',
 2010: 'co2_2010',
 2011: 'co2_2011',
 2012: 'co2_2012',
 2013: 'co2_2013',
 2014: 'co2_2014',
 2015: 'co2_2015',
 2016: 'co2_2016',
 2017: 'co2_2017',
 2018: 'co2_2018',
 2019: 'co2_2019',
 2020: 'co2_2020',
 2021: 'co2_2021',
 2022: 'co2_2022',
 2023: 'co2_2023',
 2024: 'co2_2024'}

In [23]:
rdict = {}
ddict = {}
co2dict = {}

for i in range(2015, 2025):
    rdict[('yearpower', i)] = 'energy_' + str(i)
    ddict[str(i)] = 0

for j in range(2007, PRTR_YEAR):
    co2dict[('amount', j)] = 'co2_' + str(j)

In [24]:
co2dict

{('amount', 2007): 'co2_2007',
 ('amount', 2008): 'co2_2008',
 ('amount', 2009): 'co2_2009',
 ('amount', 2010): 'co2_2010',
 ('amount', 2011): 'co2_2011',
 ('amount', 2012): 'co2_2012',
 ('amount', 2013): 'co2_2013',
 ('amount', 2014): 'co2_2014',
 ('amount', 2015): 'co2_2015',
 ('amount', 2016): 'co2_2016',
 ('amount', 2017): 'co2_2017',
 ('amount', 2018): 'co2_2018',
 ('amount', 2019): 'co2_2019',
 ('amount', 2020): 'co2_2020',
 ('amount', 2021): 'co2_2021',
 ('amount', 2022): 'co2_2022',
 ('amount', 2023): 'co2_2023',
 ('amount', 2024): 'co2_2024'}

In [25]:
np2 = np.rename(columns=rdict)

In [26]:
np3 = np2.fillna(value=ddict)

In [27]:
np4 = np3.sort_values(2020, ascending=False)

In [28]:
np4

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
249,NW100-0248923,Neurath F,Baden-Württemberg,Braunkohle,Ja,2012,1972,4211.000,in Betrieb,7,...,25468598.0,27112376.0,29463725.0,20841356.0,17221819.0,20015332.0,22326897.0,15021992.0,12629202.0,NaN
214,SN70015796,Boxberg Block Q,Sachsen,Braunkohle,Ja,2012,1979,2470.000,in Betrieb,4,...,16892980.0,17385608.0,17238224.0,16920124.0,13788512.0,14117384.0,17328260.0,11806900.0,12663172.0,12000836.0
210,BB45025564,Kraftwerk Jänschwalde Block F,Brandenburg,Braunkohle,Ja,1989,1981,3000.000,in Betrieb,6,...,19873060.0,19547884.0,18840828.0,15048036.0,11641296.0,12903344.0,12458764.0,11578228.0,9558780.0,7932020.0
169,SD666-16,Isar 2,Bayern,Kernenergie,Nein,1988,1988,1410.000,stillgelegt,1,...,11334924.0,10900504.0,11475721.0,11382289.0,11028613.0,11349943.0,11610304.0,2831770.0,NaN,NaN
170,SD666-17,Emsland A,Niedersachsen,Kernenergie,Nein,1988,1988,1336.000,stillgelegt,1,...,9211718.0,10743284.0,10913690.0,10235284.0,10832723.0,10731038.0,10715822.0,2103067.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306,BYS00925,Sammelschienen-Kraftwerk Gas- und Dampfturbinen,Bayern,Erdgas,Ja,1956,1956,112.719,in Betrieb,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
307,NW300-9046797,LEV KWG G19,Baden-Württemberg,Steinkohle,Ja,1999,1955,128.000,in Betrieb,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
309,BYS00334,Turbine 7,Bayern,Erdgas,Ja,2014,1952,54.795,in Betrieb,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
310,SD664-07,Fab Fortuna,Baden-Württemberg,Braunkohle,Ja,1948,1948,15.000,in Betrieb,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
#with pd.option_context('mode.use_inf_as_null', True):
np4 = np3.sort_values(2020, ascending=False)

In [30]:
#np4

In [31]:
#np3.sort_values("2015", ascending=False)

In [32]:
pols = pd.read_csv('../pollution/pollutants.csv')

In [33]:
pols

,year,plantid,pollutant,releases_to,amount,potency,unit_2,amount_2,pollutant2
0,2024,BB16012651,NO2,Air,745000.0,6,Tsd. t,0.745000,NO2 [Tsd. t]
1,2024,BB16012651,PM10,Air,51300.0,6,Tsd. t,0.051300,PM10 [Tsd. t]
2,2024,BB16012651,NMVOC,Air,2119000.0,3,t,2119.000000,NMVOC [t]
3,2023,BB16012651,NO2,Air,554000.0,6,Tsd. t,0.554000,NO2 [Tsd. t]
4,2023,BB16012651,PM10,Air,83100.0,6,Tsd. t,0.083100,PM10 [Tsd. t]
...,...,...,...,...,...,...,...,...,...
16281,2010,TH86012942,NO2,Air,104000.0,6,Tsd. t,0.104000,NO2 [Tsd. t]
16282,2009,TH86012942,NO2,Air,136000.0,6,Tsd. t,0.136000,NO2 [Tsd. t]
16283,2008,TH86012942,NO2,Air,150652.9,6,Tsd. t,0.150653,NO2 [Tsd. t]
16284,2007,TH86012942,CO2,Air,166000000.0,9,Mio. t,0.166000,CO2 [Mio. t]


In [34]:
#pols2

In [35]:
pols1 = pols.loc[pols['releases_to'] == "Air"].loc[pols['pollutant'] == "CO2"]

In [36]:
#pols1

In [37]:
pols2 = pols1.drop(columns=['pollutant', 'releases_to', 'potency', 'unit_2', 'amount_2', 'pollutant2'])

In [38]:
pols2a = pols2.drop_duplicates(subset=['plantid', 'year'])

In [39]:
#pols2[pols2.duplicated(['plantid', 'year'], keep=False)].sort_values("year", ascending=False)

In [40]:
pols3 = pols2a.pivot(index='plantid', columns='year', values='amount')

In [41]:
pols2a

,year,plantid,amount
55,2024,BB16018798,233000000.0
56,2023,BB16018798,285000000.0
57,2022,BB16018798,247000000.0
59,2021,BB16018798,247000000.0
61,2020,BB16018798,268000000.0
...,...,...,...
16277,2020,TH86012942,122000000.0
16278,2019,TH86012942,119000000.0
16279,2018,TH86012942,126000000.0
16280,2010,TH86012942,141000000.0


In [42]:
pols4 = pols3.reset_index()

In [43]:
co2dict2

{2007: 'co2_2007',
 2008: 'co2_2008',
 2009: 'co2_2009',
 2010: 'co2_2010',
 2011: 'co2_2011',
 2012: 'co2_2012',
 2013: 'co2_2013',
 2014: 'co2_2014',
 2015: 'co2_2015',
 2016: 'co2_2016',
 2017: 'co2_2017',
 2018: 'co2_2018',
 2019: 'co2_2019',
 2020: 'co2_2020',
 2021: 'co2_2021',
 2022: 'co2_2022',
 2023: 'co2_2023',
 2024: 'co2_2024'}

In [44]:
pols5 = pols4.rename(columns=co2dict2)

In [45]:
pols5

year,plantid,co2_2007,co2_2008,co2_2009,co2_2010,co2_2011,co2_2012,co2_2013,co2_2014,co2_2015,co2_2016,co2_2017,co2_2018,co2_2019,co2_2020,co2_2021,co2_2022,co2_2023,co2_2024
0,BB16018798,2.420000e+08,2.670000e+08,2.350000e+08,2.790000e+08,2.370000e+08,2.200000e+08,2.210000e+08,2.200000e+08,2.730000e+08,3.000000e+08,2.430000e+08,2.520000e+08,2.980000e+08,2.680000e+08,2.470000e+08,2.470000e+08,2.850000e+08,2.330000e+08
1,BB23020389,1.640000e+08,1.660000e+08,1.620000e+08,1.110000e+08,1.040000e+08,3.620000e+08,4.440000e+08,4.520000e+08,3.650000e+08,4.080000e+08,4.130000e+08,4.640000e+08,4.440000e+08,4.250000e+08,4.170000e+08,4.240000e+08,4.340000e+08,3.250000e+08
2,BB23020490,4.110000e+09,4.470000e+09,4.490000e+09,4.080000e+09,4.120000e+09,3.400000e+09,3.570000e+09,3.710000e+09,4.020000e+09,3.750000e+09,3.850000e+09,3.810000e+09,3.340000e+09,3.500000e+09,3.390000e+09,3.550000e+09,3.015000e+09,3.277000e+09
3,BB23022811,1.920000e+08,1.820000e+08,1.840000e+08,1.920000e+08,1.770000e+08,1.800000e+08,1.780000e+08,1.610000e+08,1.680000e+08,1.750000e+08,1.650000e+08,1.680000e+08,1.470000e+08,1.610000e+08,1.710000e+08,1.660000e+08,1.500000e+08,1.210000e+08
4,BB45025564,2.420000e+10,2.350000e+10,2.360000e+10,2.380000e+10,2.430000e+10,2.480000e+10,2.570000e+10,2.450000e+10,2.370000e+10,2.410000e+10,2.400000e+10,2.310000e+10,1.790000e+10,1.390000e+10,1.540000e+10,1.550000e+10,1.413400e+10,1.225700e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,ST18046,NaN,1.680000e+08,1.720000e+08,2.270000e+08,2.400000e+08,2.390000e+08,2.420000e+08,2.370000e+08,2.550000e+08,2.570000e+08,2.500000e+08,2.500000e+08,2.440000e+08,2.500000e+08,2.290000e+08,2.250000e+08,2.210000e+08,2.210000e+08
216,TH30013152,NaN,2.710333e+08,2.490000e+08,2.650000e+08,2.570000e+08,2.450000e+08,2.500000e+08,2.560000e+08,2.600000e+08,2.960000e+08,2.900000e+08,2.900000e+08,2.900000e+08,2.900000e+08,2.980000e+08,2.930000e+08,2.620000e+08,2.540000e+08
217,TH62013494,1.770000e+08,1.807979e+08,1.030000e+08,1.510000e+08,1.700000e+08,1.630000e+08,1.350000e+08,1.500000e+08,1.420000e+08,1.210000e+08,1.500000e+08,1.420000e+08,1.440000e+08,1.470000e+08,1.420000e+08,1.310000e+08,1.290000e+08,1.270000e+08
218,TH72012874,3.430000e+08,3.454339e+08,3.670000e+08,3.850000e+08,3.530000e+08,3.010000e+08,2.230000e+08,1.880000e+08,1.880000e+08,2.250000e+08,2.430000e+08,2.620000e+08,2.100000e+08,2.230000e+08,2.640000e+08,1.790000e+08,1.770000e+08,1.850000e+08


In [46]:
#final

In [47]:
final = pd.merge(np4, pols5, how='left', on="plantid")

In [48]:
final.to_csv("test.csv")
final2 = final.copy()

In [49]:
#final2 = final.rename(columns=co2dict)

In [50]:
#list(final4)

In [51]:
final2[pd.isnull(final2['plantid'])]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,...,co2_2015,co2_2016,co2_2017,co2_2018,co2_2019,co2_2020,co2_2021,co2_2022,co2_2023,co2_2024


In [52]:
final3 = final2.dropna(subset=["plantid"])

In [53]:
final4 = final3.drop(["betreiber", "eigentuemer"], axis=1)

In [54]:
col_arr = list(final4)

In [55]:
len(col_arr)

47

In [56]:
#final4.rename(columns={"2023_x": 2023}, inplace=True)

In [57]:
#final4.drop(columns="2023_y", inplace=True)

In [58]:
#final3.sort_values('plantid').reset_index(drop=True)

In [59]:
def make_to_int(df, column):
    #df[column] = df[column]
    df[column] = df[column].round().astype('Int64')

In [60]:
list(final4)[45:]

['co2_2023', 'co2_2024']

In [61]:
for i in range(17, 47):
    make_to_int(final4, col_arr[i])

In [62]:
final4.to_csv('plants_newest_nh.csv', header=False, index=False)

In [63]:
final4.loc[final4.plantid == "SN80011277"][2025]

8    7583580
Name: 2025, dtype: Int64

In [64]:
final3.loc[final3.plantid == "BYS00041"]

,plantid,plantname,federalstate,energysource,chp,latestexpanded,initialop,totalpower,state,blockcount,...,co2_2015,co2_2016,co2_2017,co2_2018,co2_2019,co2_2020,co2_2021,co2_2022,co2_2023,co2_2024
41,BYS00041,SWM HKW Nord 2 T20,Bayern,Erdgas,Ja,1991,1984,373.0,in Betrieb,3,...,2.570000e+09,2.520000e+09,2.380000e+09,2.040000e+09,2.040000e+09,1.580000e+09,1.640000e+09,1.810000e+09,2.090000e+09,1.460000e+09


In [65]:
a = int(2.990000e+10 )

In [66]:
a

29900000000

In [67]:
list(final2)

['plantid',
 'plantname',
 'federalstate',
 'energysource',
 'chp',
 'latestexpanded',
 'initialop',
 'totalpower',
 'state',
 'blockcount',
 'company',
 'betreiber',
 'eigentuemer',
 'plz',
 'ort',
 'strasse',
 'hausnr',
 'geo_lat_wgs84',
 'geo_long_wgs84',
 'activepower',
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024,
 2025,
 'co2_2007',
 'co2_2008',
 'co2_2009',
 'co2_2010',
 'co2_2011',
 'co2_2012',
 'co2_2013',
 'co2_2014',
 'co2_2015',
 'co2_2016',
 'co2_2017',
 'co2_2018',
 'co2_2019',
 'co2_2020',
 'co2_2021',
 'co2_2022',
 'co2_2023',
 'co2_2024']

In [68]:
#final2.sort_values('endop')

In [69]:
final4.columns

Index([       'plantid',      'plantname',   'federalstate',   'energysource',
                  'chp', 'latestexpanded',      'initialop',     'totalpower',
                'state',     'blockcount',        'company',            'plz',
                  'ort',        'strasse',         'hausnr',  'geo_lat_wgs84',
       'geo_long_wgs84',    'activepower',             2015,             2016,
                   2017,             2018,             2019,             2020,
                   2021,             2022,             2023,             2024,
                   2025,       'co2_2007',       'co2_2008',       'co2_2009',
             'co2_2010',       'co2_2011',       'co2_2012',       'co2_2013',
             'co2_2014',       'co2_2015',       'co2_2016',       'co2_2017',
             'co2_2018',       'co2_2019',       'co2_2020',       'co2_2021',
             'co2_2022',       'co2_2023',       'co2_2024'],
      dtype='object')

In [70]:
profit = pd.read_csv("../money/profit.csv")

In [71]:
final5 = final4.merge(profit, on="plantid", how="left")

In [72]:
final6 = final5.drop(columns=['plantname_y'])

In [73]:
final6.to_csv("plants_with_profit_v2.csv", index=False, header=False)

In [74]:
final6.columns

Index([       'plantid',    'plantname_x',   'federalstate',   'energysource',
                  'chp', 'latestexpanded',      'initialop',     'totalpower',
                'state',     'blockcount',        'company',            'plz',
                  'ort',        'strasse',         'hausnr',  'geo_lat_wgs84',
       'geo_long_wgs84',    'activepower',             2015,             2016,
                   2017,             2018,             2019,             2020,
                   2021,             2022,             2023,             2024,
                   2025,       'co2_2007',       'co2_2008',       'co2_2009',
             'co2_2010',       'co2_2011',       'co2_2012',       'co2_2013',
             'co2_2014',       'co2_2015',       'co2_2016',       'co2_2017',
             'co2_2018',       'co2_2019',       'co2_2020',       'co2_2021',
             'co2_2022',       'co2_2023',       'co2_2024',        'revenue',
               'profit'],
      dtype='object')